# TREV Gradient Benchmark: Autograd vs Parameter-Shift

Compares **autograd** (vectorized backprop) vs **parameter-shift** gradient methods.

**Benchmarks:**
1. **Accuracy** — autograd vs parameter-shift on QAOA and H2 chemistry
2. **Gradient speed** — single gradient call timing across scales
3. **Full VQE** — end-to-end optimization with convergence plots

**Circuits:**
- QAOA MaxCut (N=4..20, weighted random graphs from TREV paper)
- H2 UCCSD chemistry (4 qubits, STO-3G, full IXYZ Hamiltonian)
- Small problems use statevector as ground truth

In [ ]:
!pip install -q git+https://github.com/keunjunpark/TREV.git@real_form_autograd --force-reinstall --no-deps
!pip install -q pyscf qiskit qiskit-nature qiskit-optimization

In [ ]:
import torch, time, gc, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.optimization.gradients.autograd_gradient import (
    AutogradGradient, autograd_gradient,
)
from TREV.optimization.gradients.batch_parameter_shift import (
    BatchParameterShiftGradient, batch_gradient,
)
from TREV.optimization.optimizer import Optimizer
from TREV.optimization.optimization import minimize
from TREV.utils.maxcut import create_hamiltonian

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    free_b, total_b = torch.cuda.mem_get_info(device)
    print(f'GPU memory: {free_b/1e9:.1f} GB free / {total_b/1e9:.1f} GB total')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Helpers ──

def build_maxcut_ham(N, edges):
    """Build TREV Hamiltonian from weighted edge list."""
    hm = create_hamiltonian(N, edges)
    pauli_strings, coefficients = [], []
    for coeff, bool_list in hm:
        pauli_strings.append(''.join('Z' if b else 'I' for b in bool_list))
        coefficients.append(coeff)
    return Hamiltonian(N, pauli_strings, coefficients)

def build_random_graph(N, seed=42):
    """Random weighted complete graph (same as TREV paper)."""
    rng = np.random.RandomState(seed)
    edges = []
    for i in range(N):
        for j in range(i+1, N):
            w = rng.randint(1, 10)
            edges.append(((i, j), w))
    return edges

def build_qaoa_circuit(N, D, chi):
    """QAOA-style HEA: [RX + RZ + ring CNOT] x D."""
    c = Circuit(N, device=device)
    for _ in range(D):
        for j in range(N): c.rx(j)
        for j in range(N): c.rz(j)
        for j in range(N-1): c.cx(j, j+1)
        c.cx(N-1, 0)
    c.rank = chi
    return c

def ad_grad(theta, circuit, h, dtype=torch.cfloat):
    g, _, _ = autograd_gradient(theta, circuit, h, dtype)
    return g

def ps_grad(theta, circuit, h):
    return batch_gradient(theta, circuit, h, 8, 0, np.pi/2, 1, 0, False,
                          MeasureMethod.EFFICIENT_CONTRACTION)

def cos_sim(a, b):
    return torch.nn.functional.cosine_similarity(
        a.unsqueeze(0), b.unsqueeze(0)).item()

def bench(fn, warmup=3, repeats=5):
    for _ in range(warmup): fn()
    if device == 'cuda': torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        if device == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if device == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return np.median(times)

print('Helpers loaded.')

## 1. Accuracy: QAOA MaxCut

Autograd vs parameter-shift on QAOA circuits with weighted random graphs.

In [ ]:
torch.manual_seed(42)
rows = []

qaoa_configs = [
    # (N, chi, D)
    (4, 4, 1), (8, 8, 1), (8, 16, 1),
    (12, 8, 1), (16, 8, 1), (20, 8, 1),
]

print(f"{'config':<30} {'P':>4} {'T':>5} | {'cos(ad,ps)':>11} {'max|diff|':>10}")
print("-" * 70)

for N, chi, D in qaoa_configs:
    torch.cuda.empty_cache()
    edges = build_random_graph(N, seed=42)
    h = build_maxcut_ham(N, edges)
    c = build_qaoa_circuit(N, D, chi)
    theta = torch.randn(c.params_size, device=device)
    T = len(h.paulis)

    ga = ad_grad(theta, c, h)
    gp = ps_grad(theta, c, h)
    cos_ap = cos_sim(ga, gp)
    maxerr = (ga - gp).abs().max().item()

    label = f"MaxCut N={N:>2} chi={chi:>2} D={D}"
    print(f"{label:<30} {c.params_size:>4} {T:>5} | {cos_ap:>11.6f} {maxerr:>10.2e}")
    rows.append({'problem': 'MaxCut', 'N': N, 'chi': chi, 'D': D,
                 'P': c.params_size, 'T': T, 'cos': cos_ap, 'max_err': maxerr})

df_acc_maxcut = pd.DataFrame(rows)
df_acc_maxcut

## 2. Accuracy: TSP (QAOA)

TSP via QUBO -> Ising Hamiltonian (uses qiskit_optimization).

In [ ]:
# TSP helpers
from qiskit_optimization.applications import Tsp
from qiskit_optimization.converters import QuadraticProgramToQubo

def build_tsp_hamiltonian(n_cities, seed=42):
    """Build TSP Ising Hamiltonian via qiskit_optimization.
    Returns: (trev_hamiltonian, num_qubits, offset, qubitOp)
    """
    tsp = Tsp.create_random_instance(n_cities, seed=seed)
    qp = tsp.to_quadratic_program()
    qubo = QuadraticProgramToQubo().convert(qp)
    qubitOp, offset = qubo.to_ising()
    N = qubitOp.num_qubits

    # Convert to TREV Hamiltonian
    ps, cs = [], []
    for elm in qubitOp:
        ps.append(str(elm.paulis[0][::-1]))
        cs.append(float(elm.coeffs[0].real))
    h = Hamiltonian(N, ps, cs)
    print(f"  TSP {n_cities} cities: {N} qubits, {len(h.paulis)} terms, offset={offset:.2f}")
    return h, N, offset, qubitOp

def build_tsp_qaoa_circuit(N, D, chi):
    """QAOA ansatz for TSP: [RX + RZ + ring CNOT] x D."""
    c = Circuit(N, device=device)
    for _ in range(D):
        for j in range(N): c.rx(j)
        for j in range(N): c.rz(j)
        for j in range(N-1): c.cx(j, j+1)
        c.cx(N-1, 0)
    c.rank = chi
    return c

# Test accuracy
torch.manual_seed(42)
rows_tsp = []

tsp_configs = [
    # (n_cities, chi, D)
    (3, 8, 1),   # 9 qubits
    (4, 8, 1),   # 16 qubits
]

print(f"{'config':<35} {'P':>4} {'T':>5} | {'cos(ad,ps)':>11} {'max|diff|':>10}")
print("-" * 75)

for n_cities, chi, D in tsp_configs:
    torch.cuda.empty_cache()
    h, N, offset, _ = build_tsp_hamiltonian(n_cities, seed=42)
    c = build_tsp_qaoa_circuit(N, D, chi)
    theta = torch.randn(c.params_size, device=device)
    T = len(h.paulis)

    ga = ad_grad(theta, c, h)
    gp = ps_grad(theta, c, h)
    cos_ap = cos_sim(ga, gp)
    maxerr = (ga - gp).abs().max().item()

    label = f"TSP {n_cities}-city N={N:>2} chi={chi:>2} D={D}"
    print(f"{label:<35} {c.params_size:>4} {T:>5} | {cos_ap:>11.6f} {maxerr:>10.2e}")
    rows_tsp.append({'problem': f'TSP-{n_cities}', 'N': N, 'chi': chi, 'D': D,
                     'P': c.params_size, 'T': T, 'cos': cos_ap, 'max_err': maxerr})

df_acc_tsp = pd.DataFrame(rows_tsp)
df_acc_tsp

## 3. Accuracy: H2 & H4 Chemistry (PUCCD)

PUCCD ansatz with full IXYZ Hamiltonian via Jordan-Wigner mapping.
- H2: 4 qubits (STO-3G)
- H4: 8 qubits (STO-3G)

**Note:** `EFFICIENT_CONTRACTION` only supports Z/I, so param-shift gives wrong gradients for IXYZ.
We verify autograd against statevector finite-difference instead.

In [ ]:
import scipy.sparse.linalg as spla
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import PUCCD, HartreeFock
from qiskit.compiler import transpile as qiskit_transpile
from qiskit.circuit import ParameterExpression

jw_mapper = JordanWignerMapper()
BASIS_GATES = ['h', 'x', 'cx', 'rz', 'rx']

MOLECULES = {
    'H2':  {'atom': 'H 0 0 0; H 0 0 0.735', 'charge': 0, 'spin': 0},
    'H4':  {'atom': 'H 0 0 0; H 0 0 0.735; H 0 0 1.47; H 0 0 2.205',
            'charge': 0, 'spin': 0},
}

def setup_molecule(name):
    """Create molecular Hamiltonian + exact energy."""
    mol = MOLECULES[name]
    driver = PySCFDriver(atom=mol['atom'], charge=mol['charge'],
                         spin=mol['spin'], basis='sto3g')
    problem = driver.run()
    ns = problem.num_spatial_orbitals
    np_ = problem.num_particles
    N = 2 * ns
    nuc_rep = problem.nuclear_repulsion_energy
    qubitOp = jw_mapper.map(problem.hamiltonian.second_q_op())

    # Exact ground state
    mat = qubitOp.to_matrix(sparse=True)
    eigvals, _ = spla.eigsh(mat, k=1, which='SA')
    exact = float(eigvals[0]) + nuc_rep

    # TREV Hamiltonian (IXYZ)
    ps, cs = [], []
    for elm in qubitOp:
        ps.append(str(elm.paulis[0][::-1]))
        cs.append(float(elm.coeffs[0].real))
    h = Hamiltonian(N, ps, cs)

    print(f"  {name}: {N} qubits, {len(h.paulis)} terms, E_exact={exact:.6f} Ha")
    print(f"    has_only_zi={h.has_only_zi}")
    return h, N, ns, np_, nuc_rep, exact


def build_puccd_circuit(name, ns, np_, N, chi):
    """Build PUCCD circuit transpiled to TREV gates."""
    hf = HartreeFock(ns, np_, jw_mapper)
    ansatz = PUCCD(ns, np_, jw_mapper, initial_state=hf)
    transpiled = qiskit_transpile(ansatz, basis_gates=BASIS_GATES, optimization_level=0)

    c = Circuit(N, device=device)
    fixed_idx, fixed_vals, var_idx = [], [], []

    for inst in transpiled.data:
        gate = inst.operation
        qs = [transpiled.find_bit(q).index for q in inst.qubits]
        if gate.name == 'h':
            c.h(qs[0])
        elif gate.name == 'x':
            c.x(qs[0])
        elif gate.name == 'cx':
            c.cx(qs[0], qs[1])
        elif gate.name in ('rz', 'rx'):
            pidx = c.params_size
            if gate.name == 'rz': c.rz(qs[0])
            else: c.rx(qs[0])
            angle = gate.params[0]
            if isinstance(angle, ParameterExpression):
                var_idx.append(pidx)
            else:
                fixed_idx.append(pidx)
                fixed_vals.append(float(angle))
        elif gate.name in ('id', 'barrier'):
            pass

    c.rank = chi
    print(f"  {name} PUCCD: {c.params_size} total params "
          f"({len(var_idx)} var, {len(fixed_idx)} fixed)")
    return c, fixed_idx, fixed_vals, var_idx


# Setup molecules
mol_data = {}
for name in ['H2', 'H4']:
    mol_data[name] = setup_molecule(name)

In [ ]:
# Accuracy: autograd vs statevector finite-difference (ground truth)
from TREV.measure.contraction import contract_tensor_ring

def ev_statevector(tensor, h):
    """Exact <psi|H|psi> via statevector contraction."""
    psi = contract_tensor_ring(tensor).reshape(-1).to(torch.cfloat).to(device)
    H_mat = h.get_density_matrix().to(device)
    return (psi.conj() @ H_mat @ psi).real.item()

def fd_grad_sv(theta, circuit, h, eps=1e-4):
    """Finite-difference gradient using statevector (supports IXYZ)."""
    g = torch.zeros(theta.numel(), device=device)
    for k in range(theta.numel()):
        tp = theta.clone(); tp[k] += eps
        fp = ev_statevector(circuit.build_tensor(tp), h)
        tp[k] -= 2*eps
        fm = ev_statevector(circuit.build_tensor(tp), h)
        g[k] = (fp - fm) / (2*eps)
    return g

torch.manual_seed(42)
rows_chem = []

chem_configs = [
    ('H2', 4),   # 4 qubits
    ('H2', 8),
    ('H4', 4),   # 8 qubits — statevector still feasible (2^8=256)
    ('H4', 8),
]

print(f"{'config':<35} {'P':>4} {'T':>5} | {'cos(ad,fd)':>11} {'max|diff|':>10}")
print("-" * 75)

for mol_name, chi in chem_configs:
    torch.cuda.empty_cache()
    h, N, ns, np_, nuc_rep, exact = mol_data[mol_name]
    c, fixed_idx, fixed_vals, var_idx = build_puccd_circuit(mol_name, ns, np_, N, chi)

    theta = torch.zeros(c.params_size, device=device)
    if fixed_idx:
        theta[torch.tensor(fixed_idx, device=device)] = torch.tensor(fixed_vals, device=device)
    if var_idx:
        theta[torch.tensor(var_idx, device=device)] = torch.randn(len(var_idx), device=device) * 0.01
    T = len(h.paulis)

    ga = ad_grad(theta, c, h)
    gf = fd_grad_sv(theta, c, h)
    cos_ap = cos_sim(ga, gf)
    maxerr = (ga - gf).abs().max().item()

    label = f"{mol_name} PUCCD N={N} chi={chi}"
    print(f"{label:<35} {c.params_size:>4} {T:>5} | {cos_ap:>11.6f} {maxerr:>10.2e}")
    rows_chem.append({'problem': mol_name, 'N': N, 'chi': chi,
                      'P': c.params_size, 'T': T, 'cos': cos_ap, 'max_err': maxerr})

df_acc_chem = pd.DataFrame(rows_chem)
df_acc_chem

## 4. Gradient Speed: All Problems

Time a single gradient call for autograd vs parameter-shift.

In [ ]:
torch.manual_seed(42)
rows_speed = []

# MaxCut configs
for N, chi, D in [(4,4,1), (8,8,1), (12,8,1), (16,8,1), (20,8,1)]:
    torch.cuda.empty_cache()
    edges = build_random_graph(N, seed=42)
    h = build_maxcut_ham(N, edges)
    c = build_qaoa_circuit(N, D, chi)
    theta = torch.randn(c.params_size, device=device)
    T = len(h.paulis)

    ms_ad = bench(lambda: ad_grad(theta, c, h))
    ms_ps = bench(lambda: ps_grad(theta, c, h))
    sp = ms_ps / ms_ad

    label = f"MaxCut N={N} chi={chi}"
    print(f"{label:<30} P={c.params_size:>3} T={T:>4} | ad={ms_ad:>7.1f}ms  ps={ms_ps:>7.1f}ms  {sp:.2f}x")
    rows_speed.append({'problem': 'MaxCut', 'N': N, 'chi': chi, 'P': c.params_size,
                       'T': T, 'autograd_ms': ms_ad, 'param_shift_ms': ms_ps, 'speedup': sp})

# TSP configs
for n_cities, chi, D in [(3,8,1), (4,8,1)]:
    torch.cuda.empty_cache()
    h, N, _, _ = build_tsp_hamiltonian(n_cities, seed=42)
    c = build_tsp_qaoa_circuit(N, D, chi)
    theta = torch.randn(c.params_size, device=device)
    T = len(h.paulis)

    ms_ad = bench(lambda: ad_grad(theta, c, h))
    ms_ps = bench(lambda: ps_grad(theta, c, h))
    sp = ms_ps / ms_ad

    label = f"TSP-{n_cities} N={N} chi={chi}"
    print(f"{label:<30} P={c.params_size:>3} T={T:>4} | ad={ms_ad:>7.1f}ms  ps={ms_ps:>7.1f}ms  {sp:.2f}x")
    rows_speed.append({'problem': f'TSP-{n_cities}', 'N': N, 'chi': chi, 'P': c.params_size,
                       'T': T, 'autograd_ms': ms_ad, 'param_shift_ms': ms_ps, 'speedup': sp})

# Chemistry: autograd-only timing (param-shift doesn't support IXYZ via efficient_contraction)
print("\n--- Chemistry (autograd only — param-shift doesn't support IXYZ) ---")
for mol_name, chi in [('H2',4), ('H2',8), ('H4',4), ('H4',8)]:
    torch.cuda.empty_cache()
    h, N, ns, np_, nuc_rep, exact = mol_data[mol_name]
    c, fixed_idx, fixed_vals, var_idx = build_puccd_circuit(mol_name, ns, np_, N, chi)
    theta = torch.zeros(c.params_size, device=device)
    if fixed_idx:
        theta[torch.tensor(fixed_idx, device=device)] = torch.tensor(fixed_vals, device=device)
    T = len(h.paulis)

    ms_ad = bench(lambda: ad_grad(theta, c, h))
    label = f"{mol_name} PUCCD chi={chi}"
    print(f"{label:<30} P={c.params_size:>3} T={T:>4} | ad={ms_ad:>7.1f}ms")
    rows_speed.append({'problem': mol_name, 'N': N, 'chi': chi, 'P': c.params_size,
                       'T': T, 'autograd_ms': ms_ad, 'param_shift_ms': float('nan'), 'speedup': float('nan')})

df_speed = pd.DataFrame(rows_speed)
df_speed

## 5. Speed Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: speedup
ax = axes[0]
labels = [f"{r['problem']}\nN={r['N']}" for _, r in df_speed.iterrows()]
vals = df_speed['speedup'].values
colors = ['#2ecc71' if v >= 1 else '#e74c3c' for v in vals]
ax.bar(range(len(vals)), vals, color=colors)
ax.axhline(y=1, color='black', linestyle='--', alpha=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Speedup (param-shift / autograd)')
ax.set_title('Gradient Speedup: Autograd vs Param-Shift')
ax.grid(True, alpha=0.3, axis='y')

# Scatter: time vs P*T
ax = axes[1]
for prob in df_speed['problem'].unique():
    sub = df_speed[df_speed['problem'] == prob]
    ax.scatter(sub['P'] * sub['T'], sub['autograd_ms'], marker='o', label=f'{prob} autograd', s=60)
    ax.scatter(sub['P'] * sub['T'], sub['param_shift_ms'], marker='s', label=f'{prob} param-shift', s=60, alpha=0.6)
ax.set_xlabel('P x T (params x terms)')
ax.set_ylabel('Time (ms)')
ax.set_title('Gradient Time vs Problem Size')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Full VQE: MaxCut Convergence

End-to-end optimization comparing autograd vs parameter-shift.

In [ ]:
torch.manual_seed(42)
iters = 100

vqe_configs = [
    ('MaxCut', 8, 8, 1),
    ('MaxCut', 16, 8, 1),
    ('MaxCut', 20, 8, 1),
]

fig, axes = plt.subplots(1, len(vqe_configs), figsize=(6*len(vqe_configs), 5))
if len(vqe_configs) == 1: axes = [axes]

for idx, (prob, N, chi, D) in enumerate(vqe_configs):
    torch.cuda.empty_cache()
    edges = build_random_graph(N, seed=42)
    h = build_maxcut_ham(N, edges)
    c = build_qaoa_circuit(N, D, chi)
    theta = torch.randn(c.params_size, device=device)
    opt = Optimizer(torch.optim.Adam, {'lr': 0.05})

    # Autograd
    grad_ad = AutogradGradient(); grad_ad._verbose = False
    t0 = time.time()
    _, ev_ad, _, _ = minimize(c, theta.clone(), h, opt, grad_ad,
                               iteration=iters, best_value_method='contraction')
    t_ad = time.time() - t0

    # Param-shift
    grad_ps = BatchParameterShiftGradient(shift=np.pi/2, batch_size=None, shots=0,
        measure_method=MeasureMethod.EFFICIENT_CONTRACTION, depth=1)
    grad_ps._verbose = False
    t0 = time.time()
    _, ev_ps, _, _ = minimize(c, theta.clone(), h, opt, grad_ps,
                               iteration=iters, best_value_method='contraction')
    t_ps = time.time() - t0

    ev_ad = [float(v) if not hasattr(v, 'item') else v.item() for v in ev_ad]
    ev_ps = [float(v) if not hasattr(v, 'item') else v.item() for v in ev_ps]

    ax = axes[idx]
    ax.plot(ev_ad, label=f'Autograd ({t_ad:.1f}s)', linewidth=2)
    ax.plot(ev_ps, '--', label=f'Param-shift ({t_ps:.1f}s)', linewidth=2, alpha=0.8)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Energy')
    ax.set_title(f'MaxCut N={N} chi={chi}\nP={c.params_size} ({t_ps/t_ad:.2f}x)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    print(f"MaxCut N={N} chi={chi}: ad={t_ad/iters*1000:.0f}ms/it  ps={t_ps/iters*1000:.0f}ms/it  speedup={t_ps/t_ad:.2f}x")

plt.tight_layout()
plt.show()

## 7. Full VQE: TSP Convergence

In [ ]:
torch.manual_seed(42)
iters = 100

tsp_vqe = [(3, 8, 1), (4, 8, 1)]

fig, axes = plt.subplots(1, len(tsp_vqe), figsize=(6*len(tsp_vqe), 5))
if len(tsp_vqe) == 1: axes = [axes]

for idx, (n_cities, chi, D) in enumerate(tsp_vqe):
    torch.cuda.empty_cache()
    h, N, offset, _ = build_tsp_hamiltonian(n_cities, seed=42)
    c = build_tsp_qaoa_circuit(N, D, chi)
    theta = torch.randn(c.params_size, device=device)
    opt = Optimizer(torch.optim.Adam, {'lr': 0.05})

    # Autograd
    grad_ad = AutogradGradient(); grad_ad._verbose = False
    t0 = time.time()
    _, ev_ad, _, _ = minimize(c, theta.clone(), h, opt, grad_ad,
                               iteration=iters, best_value_method='contraction')
    t_ad = time.time() - t0

    # Param-shift
    grad_ps = BatchParameterShiftGradient(shift=np.pi/2, batch_size=None, shots=0,
        measure_method=MeasureMethod.EFFICIENT_CONTRACTION, depth=1)
    grad_ps._verbose = False
    t0 = time.time()
    _, ev_ps, _, _ = minimize(c, theta.clone(), h, opt, grad_ps,
                               iteration=iters, best_value_method='contraction')
    t_ps = time.time() - t0

    ev_ad = [float(v) if not hasattr(v, 'item') else v.item() for v in ev_ad]
    ev_ps = [float(v) if not hasattr(v, 'item') else v.item() for v in ev_ps]

    ax = axes[idx]
    ax.plot(ev_ad, label=f'Autograd ({t_ad:.1f}s)', linewidth=2)
    ax.plot(ev_ps, '--', label=f'Param-shift ({t_ps:.1f}s)', linewidth=2, alpha=0.8)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Energy')
    ax.set_title(f'TSP {n_cities}-city (N={N}) chi={chi}\nP={c.params_size} ({t_ps/t_ad:.2f}x)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    print(f"TSP-{n_cities} N={N}: ad={t_ad/iters*1000:.0f}ms/it  ps={t_ps/iters*1000:.0f}ms/it  speedup={t_ps/t_ad:.2f}x")

plt.tight_layout()
plt.show()

## 8. Full VQE: H2 & H4 Chemistry (PUCCD, autograd only)

Molecular VQE with IXYZ Hamiltonian. Only autograd supports IXYZ contractions —
parameter-shift's efficient contraction is ZI-only.

In [ ]:
torch.manual_seed(42)
iters = 200

chem_vqe = [('H2', 8), ('H4', 8)]

fig, axes = plt.subplots(1, len(chem_vqe), figsize=(7*len(chem_vqe), 5))
if len(chem_vqe) == 1: axes = [axes]

CHEM_ACCURACY = 1.6e-3  # chemical accuracy in Ha

for idx, (mol_name, chi) in enumerate(chem_vqe):
    torch.cuda.empty_cache()
    h, N, ns, np_, nuc_rep, exact = mol_data[mol_name]
    c, fixed_idx, fixed_vals, var_idx = build_puccd_circuit(mol_name, ns, np_, N, chi)

    # Init theta: fixed params + small random variational
    theta = torch.zeros(c.params_size, device=device)
    if fixed_idx:
        theta[torch.tensor(fixed_idx, device=device)] = torch.tensor(fixed_vals, device=device)
    if var_idx:
        torch.manual_seed(42)
        theta[torch.tensor(var_idx, device=device)] = torch.randn(len(var_idx), device=device) * 0.01

    # Fixed param mask — zero gradient for non-variational params
    fixed_mask = torch.zeros(c.params_size, device=device)
    if fixed_idx:
        fixed_mask[torch.tensor(fixed_idx, device=device)] = 1.0
    fv_dev = torch.tensor(fixed_vals, device=device) if fixed_vals else None

    # Manual VQE loop (autograd only, since minimize uses efficient_contraction)
    opt_torch = torch.optim.Adam([theta.clone().detach().requires_grad_(False)], lr=0.005)
    theta_vqe = theta.clone()
    energies = []

    t0 = time.time()
    for step in range(iters):
        grad, ev, _ = autograd_gradient(theta_vqe, c, h, torch.cfloat)

        # Zero gradient for fixed params
        if fixed_idx:
            grad[torch.tensor(fixed_idx, device=device)] = 0.0

        # Manual Adam step
        if step == 0:
            m = torch.zeros_like(grad)
            v = torch.zeros_like(grad)
        beta1, beta2, eps_adam, lr = 0.9, 0.999, 1e-8, 0.005
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * grad**2
        m_hat = m / (1 - beta1**(step+1))
        v_hat = v / (1 - beta2**(step+1))
        theta_vqe = theta_vqe - lr * m_hat / (v_hat.sqrt() + eps_adam)

        # Restore fixed params
        if fixed_idx and fv_dev is not None:
            theta_vqe[torch.tensor(fixed_idx, device=device)] = fv_dev

        energies.append(ev + nuc_rep)

        if step % 50 == 0 or step == iters - 1:
            print(f"  [{mol_name}] step {step:>3}: E={energies[-1]:.6f} Ha  "
                  f"(err={energies[-1]-exact:+.4f})")

    t_elapsed = time.time() - t0
    ev_ha = energies

    ax = axes[idx]
    ax.plot(ev_ha, label=f'Autograd ({t_elapsed:.1f}s)', linewidth=2)
    ax.axhline(exact, ls=':', color='black', lw=1.5, label=f'Exact ({exact:.4f} Ha)')
    ax.axhline(exact + CHEM_ACCURACY, ls='--', color='gray', lw=1, alpha=0.5,
               label='Chem. accuracy')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Energy (Ha)')
    ax.set_title(f'{mol_name} PUCCD (N={N}, chi={chi})\nP={c.params_size} ({t_elapsed/iters*1000:.0f}ms/iter)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    print(f"{mol_name}: {t_elapsed/iters*1000:.0f}ms/iter  final E={ev_ha[-1]:.6f}  exact={exact:.6f}  err={ev_ha[-1]-exact:+.4f}")

plt.tight_layout()
plt.show()

## 9. Summary Table

In [ ]:
# Combine all accuracy results
df_all_acc = pd.concat([df_acc_maxcut, df_acc_tsp, df_acc_chem], ignore_index=True)
print("=== Gradient Accuracy (autograd vs parameter-shift) ===")
print(df_all_acc[['problem', 'N', 'chi', 'P', 'T', 'cos', 'max_err']].to_string(index=False))

print("\n=== Gradient Speed ===")
print(df_speed[['problem', 'N', 'chi', 'P', 'T', 'autograd_ms', 'param_shift_ms', 'speedup']].to_string(index=False))